# Exercise - RNN Classification

In this notebook, we will perform a classification task using RNNs (i.e., a sequence to value prediction). We have hourly power consumption of households for 12 hours. Based on this, we will determine whether the power grid is strained (1) or not (0). 

Therefore, use the columns from `Hour 0` to `Hour 11` to predict the `target` column in the `power.csv` data set.

Hint1: Use Tutorial 1 for help.

Hint2: Don't forget to adjust the number of neurons in the input layers correctly. Otherwise, you will run into errors.

In [1]:
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_squared_error


# Common imports
import numpy as np
import os
import pandas as pd

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)



# Read the Dataset

In [2]:
power = pd.read_csv('power.csv')

power.head()

,Hour 0,Hour 1,Hour 2,Hour 3,Hour 4,Hour 5,Hour 6,Hour 7,Hour 8,Hour 9,Hour 10,Hour 11,target
0,2.550633,2.523400,2.582333,2.541667,2.475733,2.476233,2.455800,2.447200,2.441733,3.146133,2.661733,2.576000,1
1,1.596933,1.619567,2.473733,2.731133,2.431133,2.479667,1.690200,1.332133,1.375167,1.050900,0.585900,2.651900,1
2,0.534933,0.540467,0.575367,0.526500,0.521900,0.565333,1.426467,0.602067,0.547433,0.525067,1.270300,0.393767,0
3,1.085867,0.651233,0.634600,0.653000,0.646067,0.628400,0.611067,0.612533,0.660100,0.606067,1.471867,0.834533,0
4,0.456000,0.286300,0.310833,0.250933,0.277667,0.308633,0.610400,1.563533,1.421867,3.324400,3.207567,1.425433,1


In [3]:
power.shape

(1417, 13)

# Split the Data



In [4]:
# First 1000 days are for train
train = power.iloc[:1000]

# Remaining 417 days are for test
test = power.iloc[-417:]

In [5]:
train.shape

(1000, 13)

In [6]:
test.shape

(417, 13)

# Create Input and Target values

The first 12 columns (hourly data) will be input to predict the last column (i.e., target)

In [7]:
# The first 12 columns (from 0 to 11) are inputs

train_inputs = train.iloc[:,:12]

## Add one more dimension to make it ready for RNNs

In [8]:
#Create an additional dimension for train

train_x = np.array(train_inputs).reshape(1000,12,1)

train_x.shape 

(1000, 12, 1)

## Set the target

In [9]:
# The last column is TARGET

train_target = train.iloc[:,-1]

## Repeat for TEST

In [10]:
test.shape

(417, 13)

In [11]:
# The first 12 columns are inputs

test_inputs = test.iloc[:,:12]

In [12]:
#Create an additional dimension for test

test_x = np.array(test_inputs).reshape(417,12,1)

test_x.shape 

(417, 12, 1)

In [13]:
# The last column is TARGET

test_target = test.iloc[:,-1]

In [15]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy="most_frequent")

dummy_clf.fit(train_inputs, train_target)

DummyClassifier(strategy='most_frequent')

In [18]:
#Baseline Train Accuracy
from sklearn.metrics import accuracy_score
dummy_train_pred = dummy_clf.predict(train_inputs)

baseline_train_acc = accuracy_score(train_target, dummy_train_pred)

print('Baseline Train Accuracy: {}' .format(baseline_train_acc))

Baseline Train Accuracy: 0.505


In [19]:
#Baseline Test Accuracy
dummy_test_pred = dummy_clf.predict(test_inputs)

baseline_test_acc = accuracy_score(test_target, dummy_test_pred)

print('Baseline Test Accuracy: {}' .format(baseline_test_acc))

Baseline Test Accuracy: 0.49640287769784175


# Build a normal (cross-sectional) NN

This model assumes that the data is NOT a time-series data set. It treats the data as cross-sectional and the columns being independent of each other.

In [22]:
model = keras.models.Sequential([

    keras.layers.Flatten(input_shape=[12, 1]),
    keras.layers.Dense(12, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')  
    
])

In [24]:
np.random.seed(42)
tf.random.set_seed(42)

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.01)

model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=['accuracy'])

history = model.fit(train_inputs, train_target, epochs=50,
                    validation_data=(test_inputs, test_target))

Epoch 1/50
32/32 [==============================] - 1s 9ms/step - loss: 0.8392 - accuracy: 0.4540 - val_loss: 0.6996 - val_accuracy: 0.5036
Epoch 2/50
32/32 [==============================] - 0s 2ms/step - loss: 0.6479 - accuracy: 0.6250 - val_loss: 0.6121 - val_accuracy: 0.6978
Epoch 3/50
32/32 [==============================] - 0s 2ms/step - loss: 0.5551 - accuracy: 0.7270 - val_loss: 0.5724 - val_accuracy: 0.6859
Epoch 4/50
32/32 [==============================] - 0s 2ms/step - loss: 0.5160 - accuracy: 0.7520 - val_loss: 0.5446 - val_accuracy: 0.6882
Epoch 5/50
32/32 [==============================] - 0s 2ms/step - loss: 0.4876 - accuracy: 0.7510 - val_loss: 0.5317 - val_accuracy: 0.7002
Epoch 6/50
32/32 [==============================] - 0s 2ms/step - loss: 0.4779 - accuracy: 0.7360 - val_loss: 0.5368 - val_accuracy: 0.6906
Epoch 7/50
32/32 [==============================] - 0s 2ms/step - loss: 0.4709 - accuracy: 0.7500 - val_loss: 0.5276 - val_accuracy: 0.7266
Epoch 8/50
32/32 [==

In [27]:
# evaluate the model

scores = model.evaluate(test_inputs, test_target, verbose=0)
scores

print("%s: %.2f" % (model.metrics_names[0], scores[0]))
print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

#72.18% accuracy is much better than the baseline that is just around 50%


loss: 0.53
accuracy: 72.18%


# Build a simple RNN with one layer

In [28]:
n_steps = 12
n_inputs = 1


model = keras.models.Sequential([
    
    keras.layers.SimpleRNN(12, input_shape=[n_steps, n_inputs]),  
    keras.layers.Dense(1, activation='sigmoid')
])

In [29]:
from tensorflow.keras.callbacks import EarlyStopping
earlystop = EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='auto')
callback = [earlystop]

In [31]:
np.random.seed(42)
tf.random.set_seed(42)

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.01)

# If multiclass, use "sparse_categorical_crossentropy" as the loss function
model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=['accuracy'])


history = model.fit(train_inputs, train_target, epochs=50,
                    validation_data=(test_inputs, test_target), callbacks=callback)

Epoch 1/50
32/32 [==============================] - 1s 10ms/step - loss: 0.5973 - accuracy: 0.6870 - val_loss: 0.5293 - val_accuracy: 0.7242
Epoch 2/50
32/32 [==============================] - 0s 4ms/step - loss: 0.4950 - accuracy: 0.7530 - val_loss: 0.5225 - val_accuracy: 0.7242
Epoch 3/50
32/32 [==============================] - 0s 4ms/step - loss: 0.4812 - accuracy: 0.7620 - val_loss: 0.5816 - val_accuracy: 0.6811
Epoch 4/50
32/32 [==============================] - 0s 3ms/step - loss: 0.4818 - accuracy: 0.7600 - val_loss: 0.5276 - val_accuracy: 0.7362
Epoch 5/50
32/32 [==============================] - 0s 3ms/step - loss: 0.4636 - accuracy: 0.7660 - val_loss: 0.5325 - val_accuracy: 0.7074
Epoch 6/50
32/32 [==============================] - 0s 3ms/step - loss: 0.4717 - accuracy: 0.7600 - val_loss: 0.5142 - val_accuracy: 0.7290
Epoch 7/50
32/32 [==============================] - 0s 3ms/step - loss: 0.4621 - accuracy: 0.7700 - val_loss: 0.5012 - val_accuracy: 0.7578
Epoch 8/50
32/32 [=

In [33]:
# evaluate the model

scores = model.evaluate(test_inputs, test_target, verbose=0)
scores

print("%s: %.2f" % (model.metrics_names[0], scores[0]))
print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

# This is the best model so far.

loss: 0.50
accuracy: 74.10%


# Build a simple RNN with two or more layers

In [34]:
n_steps = 12
n_inputs = 1


model = keras.models.Sequential([
    keras.layers.SimpleRNN(12, return_sequences=True, input_shape=[n_steps, n_inputs] ),
    keras.layers.SimpleRNN(22, return_sequences=True),
    keras.layers.SimpleRNN(32), 
    keras.layers.Dense(1, activation='sigmoid')  
])


In [35]:
np.random.seed(42)
tf.random.set_seed(42)

optimizer = keras.optimizers.Nadam(learning_rate=0.01)

model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=['accuracy'])

history = model.fit(train_inputs, train_target, epochs=20,
                   validation_data = (test_inputs, test_target), callbacks=callback)

Epoch 1/20
32/32 [==============================] - 3s 18ms/step - loss: 0.5985 - accuracy: 0.6650 - val_loss: 0.5761 - val_accuracy: 0.7050
Epoch 2/20
32/32 [==============================] - 0s 6ms/step - loss: 0.5142 - accuracy: 0.7290 - val_loss: 0.6294 - val_accuracy: 0.6787
Epoch 3/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4968 - accuracy: 0.7530 - val_loss: 0.8174 - val_accuracy: 0.5612
Epoch 4/20
32/32 [==============================] - 0s 6ms/step - loss: 0.5281 - accuracy: 0.7380 - val_loss: 0.5512 - val_accuracy: 0.6978
Epoch 5/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4917 - accuracy: 0.7500 - val_loss: 0.5611 - val_accuracy: 0.6978
Epoch 6/20
32/32 [==============================] - 0s 7ms/step - loss: 0.4738 - accuracy: 0.7580 - val_loss: 0.5143 - val_accuracy: 0.7146
Epoch 7/20
32/32 [==============================] - 0s 7ms/step - loss: 0.4615 - accuracy: 0.7650 - val_loss: 0.5311 - val_accuracy: 0.7506
Epoch 8/20
32/32 [=

In [37]:
# evaluate the model

scores = model.evaluate(test_inputs, test_target, verbose=0)
scores

print("%s: %.2f" % (model.metrics_names[0], scores[0]))
print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

# This is not as good as the simple RNN

loss: 0.52
accuracy: 72.90%


# Build a LSTM with one layer

In [38]:
n_steps = 12
n_inputs = 1

model = keras.models.Sequential([
    
    keras.layers.LSTM(12, input_shape=[n_steps, n_inputs]),
    keras.layers.Dense(1, activation='sigmoid')
])

In [39]:
np.random.seed(42)
tf.random.set_seed(42)

optimizer = keras.optimizers.Nadam(learning_rate=0.01)

model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=['accuracy'])

history = model.fit(train_inputs, train_target, epochs=20,
                   validation_data = (test_inputs, test_target), callbacks=callback)

Epoch 1/20
32/32 [==============================] - 3s 18ms/step - loss: 0.5881 - accuracy: 0.6780 - val_loss: 0.5497 - val_accuracy: 0.7266
Epoch 2/20
32/32 [==============================] - 0s 5ms/step - loss: 0.4901 - accuracy: 0.7400 - val_loss: 0.5497 - val_accuracy: 0.7050
Epoch 3/20
32/32 [==============================] - 0s 5ms/step - loss: 0.4777 - accuracy: 0.7470 - val_loss: 0.5760 - val_accuracy: 0.7026
Epoch 4/20
32/32 [==============================] - 0s 5ms/step - loss: 0.4797 - accuracy: 0.7600 - val_loss: 0.5191 - val_accuracy: 0.7170
Epoch 5/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4671 - accuracy: 0.7590 - val_loss: 0.5160 - val_accuracy: 0.7122
Epoch 6/20
32/32 [==============================] - 0s 5ms/step - loss: 0.4690 - accuracy: 0.7530 - val_loss: 0.5150 - val_accuracy: 0.7098
Epoch 7/20
32/32 [==============================] - 0s 5ms/step - loss: 0.4665 - accuracy: 0.7600 - val_loss: 0.5108 - val_accuracy: 0.7578
Epoch 8/20
32/32 [=

In [41]:
# evaluate the model

scores = model.evaluate(test_inputs, test_target, verbose=0)
scores

print("%s: %.2f" % (model.metrics_names[0], scores[0]))
print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

# Best model so far.

loss: 0.50
accuracy: 75.30%


# Build a LSTM with two or more layers

In [42]:
n_steps = 12
n_inputs = 1

model = keras.models.Sequential([
    keras.layers.LSTM(12, return_sequences=True, input_shape=[n_steps, n_inputs]),
    keras.layers.LSTM(12, return_sequences=True),
    keras.layers.LSTM(12),
    keras.layers.Dense(1, activation='sigmoid')
])

In [43]:
np.random.seed(42)
tf.random.set_seed(42)

optimizer = keras.optimizers.Nadam(learning_rate=0.01)

model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=['accuracy'])

history = model.fit(train_inputs, train_target, epochs=20,
                   validation_data = (test_inputs, test_target), callbacks=callback)

Epoch 1/20
32/32 [==============================] - 6s 42ms/step - loss: 0.6243 - accuracy: 0.6400 - val_loss: 0.6167 - val_accuracy: 0.6811
Epoch 2/20
32/32 [==============================] - 0s 13ms/step - loss: 0.5365 - accuracy: 0.7360 - val_loss: 0.6421 - val_accuracy: 0.6715
Epoch 3/20
32/32 [==============================] - 0s 13ms/step - loss: 0.5140 - accuracy: 0.7460 - val_loss: 0.7512 - val_accuracy: 0.5827
Epoch 4/20
32/32 [==============================] - 0s 12ms/step - loss: 0.5044 - accuracy: 0.7380 - val_loss: 0.5456 - val_accuracy: 0.7122
Epoch 5/20
32/32 [==============================] - 0s 12ms/step - loss: 0.4795 - accuracy: 0.7600 - val_loss: 0.5627 - val_accuracy: 0.7050
Epoch 6/20
32/32 [==============================] - 0s 13ms/step - loss: 0.4766 - accuracy: 0.7530 - val_loss: 0.5202 - val_accuracy: 0.7146
Epoch 7/20
32/32 [==============================] - 0s 12ms/step - loss: 0.4660 - accuracy: 0.7610 - val_loss: 0.5054 - val_accuracy: 0.7554
Epoch 8/20
32

In [45]:
# evaluate the model

scores = model.evaluate(test_inputs, test_target, verbose=0)
scores

print("%s: %.2f" % (model.metrics_names[0], scores[0]))
print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

# Worse NN model

loss: 0.53
accuracy: 71.94%


# Build a GRU with one layer

In [46]:
n_steps = 12
n_inputs = 1

model = keras.models.Sequential([
    keras.layers.GRU(20, input_shape=[n_steps, n_inputs]),
    keras.layers.Dense(1, activation='sigmoid')
])

In [47]:
np.random.seed(42)
tf.random.set_seed(42)

optimizer = keras.optimizers.Nadam(learning_rate=0.01)

model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=['accuracy'])

history = model.fit(train_inputs, train_target, epochs=20,
                   validation_data = (test_inputs, test_target), callbacks=callback)

Epoch 1/20
32/32 [==============================] - 2s 17ms/step - loss: 0.5724 - accuracy: 0.6810 - val_loss: 0.5525 - val_accuracy: 0.7458
Epoch 2/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4786 - accuracy: 0.7390 - val_loss: 0.5394 - val_accuracy: 0.7074
Epoch 3/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4727 - accuracy: 0.7500 - val_loss: 0.5988 - val_accuracy: 0.6978
Epoch 4/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4757 - accuracy: 0.7600 - val_loss: 0.5286 - val_accuracy: 0.7122
Epoch 5/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4676 - accuracy: 0.7550 - val_loss: 0.5206 - val_accuracy: 0.7098
Epoch 6/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4678 - accuracy: 0.7510 - val_loss: 0.5109 - val_accuracy: 0.7098
Epoch 7/20
32/32 [==============================] - 0s 6ms/step - loss: 0.4629 - accuracy: 0.7540 - val_loss: 0.5018 - val_accuracy: 0.7578
Epoch 8/20
32/32 [=

In [49]:
# evaluate the model

scores = model.evaluate(test_inputs, test_target, verbose=0)
scores

print("%s: %.2f" % (model.metrics_names[0], scores[0]))
print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

loss: 0.51
accuracy: 73.14%


# Build a GRU with two or more layers

In [50]:
n_steps = 12
n_inputs = 1

model = keras.models.Sequential([
    keras.layers.GRU(10, return_sequences=True, input_shape=[n_steps, n_inputs]),
    keras.layers.GRU(10, return_sequences=True),
    keras.layers.GRU(10),
    keras.layers.Dense(1, activation='sigmoid')
])

In [51]:
np.random.seed(42)
tf.random.set_seed(42)

optimizer = keras.optimizers.Nadam(learning_rate=0.01)

model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=['accuracy'])

history = model.fit(train_inputs, train_target, epochs=20,
                   validation_data = (test_inputs, test_target), callbacks=callback)

Epoch 1/20
32/32 [==============================] - 6s 47ms/step - loss: 0.5928 - accuracy: 0.6800 - val_loss: 0.5723 - val_accuracy: 0.7266
Epoch 2/20
32/32 [==============================] - 0s 14ms/step - loss: 0.5027 - accuracy: 0.7280 - val_loss: 0.5664 - val_accuracy: 0.7074
Epoch 3/20
32/32 [==============================] - 0s 13ms/step - loss: 0.4881 - accuracy: 0.7420 - val_loss: 0.6698 - val_accuracy: 0.6403
Epoch 4/20
32/32 [==============================] - 0s 14ms/step - loss: 0.4948 - accuracy: 0.7540 - val_loss: 0.5293 - val_accuracy: 0.7074
Epoch 5/20
32/32 [==============================] - 0s 14ms/step - loss: 0.4684 - accuracy: 0.7600 - val_loss: 0.5506 - val_accuracy: 0.7170
Epoch 6/20
32/32 [==============================] - 0s 15ms/step - loss: 0.4655 - accuracy: 0.7570 - val_loss: 0.5121 - val_accuracy: 0.7242
Epoch 7/20
32/32 [==============================] - 0s 14ms/step - loss: 0.4575 - accuracy: 0.7600 - val_loss: 0.4954 - val_accuracy: 0.7530
Epoch 8/20
32

In [53]:
# evaluate the model

scores = model.evaluate(test_inputs, test_target, verbose=0)
scores

print("%s: %.2f" % (model.metrics_names[0], scores[0]))
print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

# This final model is the best, edging LSTM with one layer

loss: 0.50
accuracy: 75.78%
